# Course 1: Lidar
## Part 4: Working with Real PCD
#### By Jonathan L. Moran (jonathan.moran107@gmail.com)
From the Sensor Fusion Nanodegree programme offered at Udacity.

## Objectives

## 1. Introduction

### 1.1. The `CityBlock` Scene

In this lesson, we make use of a new simulated driving environment: the `CityBlock` scene. In this scene are multiple obstacles captured by a real LiDAR sensor mounted to an actual "self-driving car". The scene itself captures a four-way intersection with moderate traffic density with a diverse set of obstacles occupying the `CityBlock` scene, e.g., cars, road signs, buildings, etc.

### 1.2 Filtering with Point Cloud Library (PCL)

Individual point cloud "scans", depending on the LiDAR sensor specifications, contain hundreds of thousands of data points. Each of these scans are recorded at very high intervals, typically between 10-30 frames per second. In order to process each one of these scans in "real-time", many system engineers choose to [downsample](https://en.wikipedia.org/wiki/Downsampling_(signal_processing)) the LiDAR point clouds. By intentionally filtering / removing many of the LiDAR points in each scan, we can "speed up" processing time and drastically reduce the file sizes of the point clouds. The two techniques for downsampling we will discuss here are _grid voxelisation_ and _region-based filtering_. 

[Voxelisation](https://pointclouds.org/documentation/tutorials/voxel_grid.html) is the process of fitting 3D geometry onto the captured scene in order to perform point manipulation inside each fitted region. For example, we can "represent" the flat surface of a table as a simple 3D "box". Using the 3D shape to define the object dimensions, we are able to constrain the points scanned off the table's surface then reduce the points in the "voxel" area using e.g., centroid approximation, which preserves only one centre point inside each "leaf" formed by the voxel grid. Here, a "leaf" refers to a pre-defined area (e.g., $1 cm$) in which the voxel grid is discretised (i.e., divided up). An example of downsampling using the Point Cloud Library (PCL) [`pcl::VoxelGrid`](https://pointclouds.org/documentation/classpcl_1_1_voxel_grid_3_01pcl_1_1_p_c_l_point_cloud2_01_4.html#ad3efe8cb07386b291c88c5178e8b135d) filter is visualised in [this YouTube video](https://www.youtube.com/watch?v=YHR6_OIxtFI&t) where the original scan (left) is reduced (right) with a centroid approximation-based leaf method. This is a type of uniformly-distributed voxelisation method — when strategically configured — allows the shape of the objects captured to be relatively preserved. However, this method may not be practical as it requires "foresight" into the object geometry being scanned to strategically determine the appropriate "leaf" size(s) that will result in a desirable point density while preserving the underlying object representation. While there are more-advanced techniques for voxelisation (e.g., deep learning-based [PointNet](https://github.com/charlesq34/pointnet), [PointNet++](https://stanford.edu/~rqi/pointnet2/) and [VoxelNet](https://arxiv.org/abs/1711.06396)), we will leave the exploration into those methods for you to consider.

[Region-based filtering]() is another category of techniques for reducing the number of total points in a LiDAR scan. With region-based filtering, we have the ability to "apply" multiple geometric representations across sub-regions of a single unstructured point cloud. Using the statistical multi-scale interest region-based extraction [] method, we can employ a data-driven approach to selecting the location(s) of our "interest regions" amd the "support radius" value(s) they are defined by. Unlike with the uniformly-distributed voxel grid method above, this mulit-scale region-based method allows us to "break apart" the point cloud into sub-regions, each with varying "geometries". The multi-scale interest region method allows us to e.g., reduce redundancy in areas with little shape variation, and better represent sparsity in these local "neighbourhoods" of points, by guiding the selection process with locally-informed features (called "descriptors") within the sub-regions. We can use the Unnikrishnan et al. (2008) method natively in the Point Cloud Library (PCL) by creating a [`pcl::StatisticalMultiscaleInterestRegionExtraction`](https://pointclouds.org/documentation/classpcl_1_1_statistical_multiscale_interest_region_extraction.html) class instance.

### 

## 2. Programming Task

### The `CityBlock` Scene

#### E1.4.0: `cityBlock`

In this section we create the `cityBlock` function inside [`environment.cpp`]() which performs the following steps:
1. Create a point processor instance that stores `pcl::PointXYZI` data;
2. Load the PCD file and render its data onto the PCL Viewer.


##### The `cityBlock` function

```cpp
// From J. Moran's `src/environment.cpp`:
// Credit: 
```

```cpp
// In `src/environment.cpp`:

void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    /** E1.4.0: Render the `CityBlock` Scene. **/
    // Creating a new point processor (stores Intensity values)
    ProcessPointClouds<
        pcl::PointXYZI
    > *pointProcessorI = new ProcessPointClouds<pcl::PointXYZI>();
    // Loading the `CityBlock` point cloud data
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr inputCloud = pointProcessorI->loadPcd(
        "../src/sensors/data/pcd/data_1/0000000000.pcd"
    );
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        inputCloud,
        "inputCloud — City Block Scan"
    );
}
```

##### Testing the `cityBlock` function

With the `cityBlock` function defined, we will now call it from inside the `main` programme function instead of the previous `simpleHighway` scene. 

```cpp
// In `src/environment.cpp`:

int main() {
    // ..
    /** E1.1.0: Create 3D highway scene. **/
    // simpleHighway(viewer);
    /** E1.4.0: Render the `CityBlock` Scene. **/
    cityBlock(viewer);
}
```

To run the `cityBlock` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

With the point cloud data (PCD) file loaded from its directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`), we obtain the following output:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
```

<img src="figures/2024-07-06-Figure-1-cityBlock-Rendering.png" alt="Figure 1. The cityBlock scene — Rendered with Point Cloud Library (PCL)." height="70%" width="70%">

$$\textrm{Figure 1. The cityBlock scene — Rendered with Point Cloud Library (PCL).}$$

#### E1.4.1: `FilterCloud` with `pcl::VoxelGrid`

##### The `pcl::VoxelGrid` method

```cpp
// From J. Moran's `src/processPointClouds.cpp`:
// Credit:
```

```cpp
// In `src/processPointClouds.cpp`:

template<typename PointT> typename pcl::PointCloud<
    PointT
>::Ptr ProcessPointClouds<PointT>::FilterCloud(
    typename pcl::PointCloud<PointT>::Ptr cloud, 
    float filterRes, 
    Eigen::Vector4f minPoint, 
    Eigen::Vector4f maxPoint
) {
    // Time segmentation process
    auto startTime = std::chrono::steady_clock::now();
    /** E1.4.1: Filtering the point cloud. **/
    // TODO:: Fill in the function to do voxel grid point reduction and region based filtering
    typename pcl::PointCloud<PointT>::Ptr cloudFiltered(
        new pcl::PointCloud<PointT>
    );
    // Creating the voxel-based filtering object
    pcl::VoxelGrid<PointT> vg;
    vg.setInputCloud(cloud);
    // Specifying the leaf size / "cell" dimensions
    vg.setLeafSize(filterRes, filterRes, filterRes);
    vg.filter(*cloudFiltered);
    auto endTime = std::chrono::steady_clock::now();
    auto elapsedTime = std::chrono::duration_cast<
        std::chrono::milliseconds
    >(endTime - startTime);
    std::cout << "filtering took "
              << elapsedTime.count() << " milliseconds\n";
    return cloudFiltered;
}
```

##### Testing the `pcl::VoxelGrid` method

To apply the filtering function, call the `FilterCloud` function we defined above with the `pointProcessorI` instance from inside the `cityBlock` function, as follows:

```cpp
// In `src/environment.cpp`:

void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    /** E1.4.1: Filtering with `pcl::VoxelGrid` **/
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr filterCloud = pointProcessorI->FilterCloud(
        inputCloud,
        0.2f,
        Eigen::Vector4f(0.0, 0.0, 0.0, 1.0), // minPoint; ignore for now.
        Eigen::Vector4f(0.0, 0.0, 0.0, 1.0) // maxPoint; ignore for now.
    );
    std::cerr << "Loaded " << filterCloud->points.size()
              << " data points from filtered cloud\n";
    // ..
}
```

To run the `cityBlock` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

To view the filtered point cloud, you must pass the `filterCloud` instance into the `renderPointCloud` function on `line 108` inside the `cityBlock` function:

```cpp
void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        filterCloud, // inputCloud; replace to view filtered cloud instead
        "inputCloud — City Block Scan (filtered)"
    );
}
```

With the _voxel grid filtered_ point cloud data (PCD) file loaded from its directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`), we obtain the following output:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
filtering took 7 milliseconds
Loaded 23273 data points from filtered cloud
```

<img src="figures/2024-07-06-Figure-2-cityBlock-Rendering-Voxel-Grid-Filtering-Comparison.png" alt="Figure 2. The `cityBlock` scene — Unmodified (left) versus Filtered (right) point cloud downsampled with Voxel Grid method using Point Cloud Library (PCL)." height="70%" width="70%">

$$\textrm{Figure 2. The `cityBlock` scene — Unmodified (left) versus Filtered (right) point cloud downsampled with Voxel Grid method using Point Cloud Library (PCL).}$$

To produce the above downsampled point cloud, a voxel "cell" dimension of `0.2f` was provided (the input argument `filterRes` to the `FilterCloud` function).

#### E1.4.2(a): `FilterCloud` with `pcl::CropBox`

In this section we "downsample" our given point cloud scan by selectively removing points that fall within a defined region.


For our test case, we want to utilise `pcl::CropBox` to define two regions of interest: the first, the "scene", will be the total point cloud "area" we wish to preserve — all points _outside_ this defined region will be eliminated. The other area of interest, the ego-vehicle "roof", will be the area blanketing the roof of the ego-vehicle — all points _inside_ this defined region will be eliminated.

Let's start with the first region, the "scene". Similar to the above E1.4.1, we make use of the `FilterCloud` method to _filter_ (eliminate) points in the cloud which lie _outside_ the defined region of interest. To do so, we pass in non-zero values for the `minPoint` and `maxPoint` input arguments. These two `Eigen::Vector4f` coordinate pairs define the rectangular area of the total point cloud area we wish to preserve.

##### The `pcl::CropBox` method

```cpp
// From J. Moran's `src/processPointClouds.cpp`:
// Credit:
```

```cpp
// In `src/processPointClouds.cpp`:

template<typename PointT> typename pcl::PointCloud<
    PointT
>::Ptr ProcessPointClouds<PointT>::FilterCloud(
    typename pcl::PointCloud<PointT>::Ptr cloud, 
    float filterRes, 
    Eigen::Vector4f minPoint, 
    Eigen::Vector4f maxPoint
) {
    // ..
    /** E1.4.2(a): Filtering the point cloud with `pcl::CropBox`. **/
    typename pcl::PointCloud<PointT>::Ptr cloudRegion(new pcl::PointCloud<PointT>);
    // Defining the first region: the area of points to preserve
    pcl::CropBox<PointT> regionPreserved(true);
    regionPreserved.setMin(minPoint);
    regionPreserved.setMax(maxPoint);
    regionPreserved.setInputCloud(cloudFiltered);
    // Cropping the point cloud to the desired region (the "scene")
    regionPreserved.filter(*cloudRegion);
    // ..
}
```

##### Testing the `pcl::CropBox` method

In order to make use of the region-based cropping method, we must assign the two input arguments `minPoint` and `maxPoint` values associated with the region we wish to "crop". These two variables are defined as `Eigen::Vector4f` types, where each represents a 3D point.

For our test case, we want to utilise `pcl::CropBox` to define the first region of interest as the total area of the point cloud which we are interested in. We hand-select these values such that all anticipated scene details remain preserved, while eliminating points which are either "too far" away from the ego-vehicle to be useful for processing, or points which are unlikely to provide significant information to act upon (e.g., points belonging to adjacent buildings or traffic from multiple lanes away). 

Side note: this is just for demonstration purposes, and not design advice for actual AVs. We leave that for you to reason about.



```cpp
// From J. Moran's `src/environment.cpp`:
// Credit:
```

```cpp
// In `src/environment.cpp`:

void cityBlock{
    // ..
    /** E1.4.2(a): Filtering the point cloud with `pcl::CropBox`. **/
    // NOTE: choosing non-zero valued vectors for `minPoint`, `maxPoint`;
    // These define the area of the region we wish to preserve. 
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr regionCloud = pointProcessorI->FilterCloud(
        inputCloud,
        0.2f,
        Eigen::Vector4f(-1.5, -1.7, -1.0, 1),
        Eigen::Vector4f(2.6, 1.7, -0.4, 1)
    );
    // ..
}
```

To run the `cityBlock` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

To view the region-based filtered point cloud, you must pass the `regionCloud` instance into the `renderPointCloud` function on `line 124` inside the `cityBlock` function:

```cpp
void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        regionCloud, // Alternatives: `inputCloud` or `filterCloud`
        "regionCloud — City Block Scan (filtered with region-based method)"
    );
    // ..
}
```

The _region-based filtered_ point cloud (`regionCloud`) contains data (in `.pcd` format) from the file loaded at the directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`). The following console output is obtained:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
filtering took 7 milliseconds
filtering took 32 milliseconds
Loaded 15 data points from filtered cloud
```

<img src="figures/2024-07-06-Figure-2-cityBlock-Rendering-Region-Based-Filtering.png" alt="Figure 3. The 'cityBlock' scene — Filtered point cloud downsampled with Region-Based Filtering with Point Cloud Library (PCL).">

$$\textrm{Figure 3. The 'cityBlock' scene — Filtered point cloud downsampled with Region-Based Filtering with Point Cloud Library (PCL).}$$

## 3. Closing Remarks

#### Alternatives
* Render a different scene in `E1.4.0` (choose from the available `.pcd` files inside the [`"../src/sensors/data/pcd/data_1/"`]() sub-directory);
* Experiment with different voxel "cell" dimensions in `E1.4.1` (i.e., choose a different value for the `filterRes` input arugument and observe its impact on the number of points in the filtered cloud as compared to the original);

#### Extensions of task

## 4. Future Work

* ⬜️
* ✅

## Credits

This assignment was prepared by Aaron Brown and Michael Maile of Mercedes-Benz Research & Development of North America (MBRDNA), 2021 (link [here](https://learn.udacity.com/nanodegrees/nd313/)).


References
* [] Unnikrishnan, R., Hebert, M. Multi-scale interest regions from unorganized point clouds. Workshop on Search in 3D (S3D), IEEE Conference on Computer Vision and Pattern Recognition (CVPR). June 2008. [doi:10.1109/CVPRW.2008.4563030](http://dx.doi.org/10.1109/CVPRW.2008.4563030).


Helpful resources:
* 